# Google Jobs Basic UI (Split Workflow)
Run cells in order: setup -> UI -> scrape -> display/save.

In [1]:
# 1) Setup modules (UV-first, pip fallback)
import shutil
import subprocess
import sys

_PKGS = ["ipywidgets", "feedparser", "pandas", "psutil", "httpx"]

if shutil.which("uv"):
    print("Using uv pip to install deps in current Python...")
    subprocess.run(["uv", "pip", "install", "--python", sys.executable, *_PKGS], check=False)
else:
    print("uv not found, using pip fallback...")
    subprocess.run([sys.executable, "-m", "pip", "install", *_PKGS], check=False)

import os
import re
import urllib.parse
from datetime import datetime
from typing import Dict, List, Tuple

import feedparser
import httpx
import pandas as pd
import ipywidgets as w
from IPython.display import display, HTML

AGE_QUERY = {
    "Any": "",
    "Past 24h": "posted today OR since yesterday",
    "Past 3 days": "posted in last 3 days",
    "Past 7 days": "posted in last week",
    "Past 30 days": "posted in last month",
}

EXP_HINT = {
    "Any": "",
    "0-1 years": "entry level OR junior OR fresher OR intern",
    "1-3 years": "1 year OR 2 years OR 3 years OR junior",
    "3-5 years": "3 years OR 4 years OR 5 years",
    "5+ years": "senior OR lead OR staff OR principal OR 5+ years",
}

GEO_SCOPE = {
    "India": {"hl": "en-IN", "gl": "IN", "ceid": "IN:en"},
    "Global": {"hl": "en", "gl": "US", "ceid": "US:en"},
}

BOARD_FILTER = "site:linkedin.com OR site:indeed.com OR site:glassdoor.com OR site:naukri.com OR site:wellfound.com OR site:workatastartup.com"

def _allowed_domains_from_board_filter(board_filter: str) -> List[str]:
    return sorted({m.group(1).lower() for m in re.finditer(r"site:([^\s]+)", board_filter)})

ALLOWED_DOMAINS = _allowed_domains_from_board_filter(BOARD_FILTER)

LAST_ROWS: List[Dict] = []
LAST_RSS_URL = ""
LAST_QUERY = ""
print("Setup complete.")

Using uv pip to install deps in current Python...
Setup complete.


In [6]:
# 2) Basic UI for inputs
role = w.Text(value="data analyst", description="Role", layout=w.Layout(width="460px"))
location = w.Text(value="remote", description="Location", layout=w.Layout(width="460px"))
age = w.Dropdown(options=list(AGE_QUERY.keys()), value="Past 7 days", description="Age")
experience = w.Dropdown(options=list(EXP_HINT.keys()), value="Any", description="Experience")
geography = w.Dropdown(options=list(GEO_SCOPE.keys()), value="India", description="Audience")
limit = w.IntSlider(value=20, min=5, max=50, step=5, description="Limit")

source = w.Dropdown(
    options=[
        ("Google News RSS (fallback)", "google_news_rss"),
        ("SerpAPI Google Jobs (requires SERPAPI_API_KEY)", "serpapi_google_jobs"),
    ],
    value="google_news_rss",
    description="Source",
    layout=w.Layout(width="520px"),
)
strict_job_boards_only = w.Checkbox(value=True, description="Only job-board/portal URLs")

remote_only = w.Checkbox(value=True, description="Remote-only intent")
associate_focus = w.Checkbox(value=False, description="Associate / junior focus")
india_friendly = w.Checkbox(value=False, description="India-friendly hiring")
visa_sponsorship = w.Checkbox(value=False, description="Visa sponsorship")

display(w.VBox([
    w.HTML("<h3>Input Parameters</h3>"),
    source,
    strict_job_boards_only,
    role,
    location,
    w.HBox([age, experience]),
    w.HBox([geography, limit]),
    w.HBox([remote_only, associate_focus]),
    w.HBox([india_friendly, visa_sponsorship]),
    w.HTML("<div style='font-size:12px;opacity:0.8'>Allowed domains: " + ", ".join(ALLOWED_DOMAINS) + "</div>"),
]))
print("Adjust inputs above, then run the scrape cell.")

Adjust inputs above, then run the scrape cell.


In [9]:
# 3) Scraping part

def build_query(
    role_v: str,
    location_v: str,
    age_bucket: str,
    exp_bucket: str,
    remote_only_v: bool = False,
    associate_focus_v: bool = False,
    india_friendly_v: bool = False,
    visa_sponsorship_v: bool = False,
) -> str:
    r = (role_v or "data analyst").strip()
    l = (location_v or "remote").strip()
    age_part = AGE_QUERY.get(age_bucket, "")
    exp_part = EXP_HINT.get(exp_bucket, "")

    remote_hint = "remote OR work from home OR wfh OR distributed" if remote_only_v else ""
    associate_hint = "associate OR junior OR entry level OR fresher" if associate_focus_v else ""
    india_hint = "India OR Indian candidates OR IST timezone" if india_friendly_v else ""
    visa_hint = "visa sponsorship OR sponsorship available" if visa_sponsorship_v else ""

    parts = [f"({r})", "jobs", f"({l})", age_part, exp_part, remote_hint, associate_hint, india_hint, visa_hint, BOARD_FILTER]
    return " ".join([p for p in parts if p]).strip()


def _url_domain(u: str) -> str:
    try:
        return (urllib.parse.urlparse(u).netloc or "").lower()
    except Exception:
        return ""


def _is_allowed_job_board_url(u: str) -> bool:
    d = _url_domain(u)
    if not d:
        return False
    if d.endswith("news.google.com"):
        return False
    return any(d == dom or d.endswith("." + dom) for dom in ALLOWED_DOMAINS)


def fetch_google_news_jobs(q: str, geography_v: str, limit_v: int = 20, strict_job_boards_only_v: bool = True):
    geo = GEO_SCOPE.get(geography_v, GEO_SCOPE["India"])
    rss_url = (
        "https://news.google.com/rss/search?q="
        + urllib.parse.quote_plus(q)
        + f"&hl={geo['hl']}&gl={geo['gl']}&ceid={geo['ceid']}"
    )
    feed = feedparser.parse(rss_url)
    rows = []
    seen = set()

    for entry in feed.entries:
        title = (getattr(entry, "title", "") or "").strip()
        link = (getattr(entry, "link", "") or "").strip()
        published = (getattr(entry, "published", "") or "").strip()
        if not title or not link or link in seen:
            continue
        seen.add(link)

        # Google News RSS commonly returns news.google.com/article links.
        # In strict mode, keep only direct job-board/portal URLs.
        if strict_job_boards_only_v and not _is_allowed_job_board_url(link):
            continue

        source_match = re.search(r"-\s*([^\-]+)$", title)
        source_hint = source_match.group(1).strip() if source_match else (_url_domain(link) or "google-news")
        rows.append({"title": title, "url": link, "published": published, "source_hint": source_hint, "query": q, "source": "google_news_rss"})
        if len(rows) >= int(limit_v):
            break

    return rows, rss_url, q


def fetch_serpapi_google_jobs(q: str, location_v: str, limit_v: int = 20):
    api_key = (os.environ.get("SERPAPI_API_KEY") or "").strip()
    if not api_key:
        raise RuntimeError("SERPAPI_API_KEY is not set (needed for SerpAPI Google Jobs).")

    params = {
        "engine": "google_jobs",
        "q": q,
        "location": (location_v or "").strip() or "Remote",
        "hl": "en",
        "api_key": api_key,
    }
    base_url = "https://serpapi.com/search.json"
    with httpx.Client(timeout=30) as client:
        r = client.get(base_url, params=params)
        r.raise_for_status()
        data = r.json()

    results = data.get("jobs_results") or []
    rows = []
    for it in results:
        title = (it.get("title") or "").strip()
        company = (it.get("company_name") or "").strip()
        job_loc = (it.get("location") or "").strip()
        published = (it.get("detected_extensions") or {}).get("posted_at") or ""
        job_id = it.get("job_id")

        url_direct = ""
        for rl in (it.get("related_links") or []):
            u = (rl.get("link") or "").strip()
            if u:
                url_direct = u
                break
        if not url_direct and job_id:
            url_direct = f"https://serpapi.com/search.json?engine=google_jobs_listing&job_id={job_id}&api_key={api_key}"

        if not title:
            continue

        rows.append({
            "title": title,
            "company": company,
            "location": job_loc,
            "published": published,
            "url": url_direct,
            "source_hint": "serpapi/google_jobs",
            "query": q,
            "source": "serpapi_google_jobs",
        })
        if len(rows) >= int(limit_v):
            break

    safe_params = {k: v for k, v in params.items() if k != "api_key"}
    return rows, base_url + "?" + urllib.parse.urlencode(safe_params), q


q = build_query(
    role.value,
    location.value,
    age.value,
    experience.value,
    remote_only.value,
    associate_focus.value,
    india_friendly.value,
    visa_sponsorship.value,
)

try:
    if source.value == "serpapi_google_jobs":
        LAST_ROWS, LAST_RSS_URL, LAST_QUERY = fetch_serpapi_google_jobs(q, location.value, limit.value)
    else:
        LAST_ROWS, LAST_RSS_URL, LAST_QUERY = fetch_google_news_jobs(q, geography.value, limit.value, strict_job_boards_only.value)
except Exception as e:
    LAST_ROWS, LAST_RSS_URL, LAST_QUERY = [], "", q
    print("Scrape failed:", str(e))

print(f"Query: {LAST_QUERY}")
print(f"URL: {LAST_RSS_URL}")
print(f"Found {len(LAST_ROWS)} results")

Scrape failed: name 'source' is not defined
Query: (data analyst) jobs (remote) posted in last 3 days 1 year OR 2 years OR 3 years OR junior remote OR work from home OR wfh OR distributed associate OR junior OR entry level OR fresher India OR Indian candidates OR IST timezone site:linkedin.com OR site:indeed.com OR site:glassdoor.com OR site:naukri.com OR site:wellfound.com OR site:workatastartup.com
URL: 
Found 0 results


In [8]:
# 4) Display + save
if not LAST_ROWS:
    print("No results yet. Run the scraping cell first.")
else:
    df = pd.DataFrame(LAST_ROWS)
    html_rows = []
    for i, r in enumerate(LAST_ROWS, 1):
        html_rows.append(f"<tr><td>{i}</td><td><a href='{r['url']}' target='_blank' rel='noopener'>{r['title']}</a></td><td>{r['source_hint']}</td><td>{r['published']}</td></tr>")
    table = "<table style='border-collapse:collapse;width:100%'><thead><tr><th style='text-align:left'>#</th><th style='text-align:left'>Title</th><th style='text-align:left'>Source</th><th style='text-align:left'>Published</th></tr></thead><tbody>" + "".join(html_rows) + "</tbody></table>"
    display(HTML(table))
    os.makedirs("out", exist_ok=True)
    stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    csv_path = os.path.join("out", f"google_jobs_basic_{stamp}.csv")
    json_path = os.path.join("out", f"google_jobs_basic_{stamp}.json")
    df.to_csv(csv_path, index=False)
    df.to_json(json_path, orient="records", force_ascii=False, indent=2)
    print("Saved:")
    print("-", csv_path)
    print("-", json_path)

#,Title,Source,Published
1,The 10 Most In-Demand IT Jobs in India for 2026 (And How to Get Them) - talentsprint.com,talentsprint.com,"Fri, 26 Dec 2025 08:00:00 GMT"
2,"What is Cyber Security?: Subjects, Course Fees, Admission 2026, Career Options - Shiksha.com",Shiksha.com,"Tue, 24 Mar 2026 07:00:00 GMT"
3,Which Jobs Will Be Most in Demand in Malaysia over the next 10 Years? - Y-Axis Overseas Careers,Axis Overseas Careers,"Thu, 11 Dec 2025 08:00:00 GMT"
4,"Highest Paying Jobs in India: Roles, Salaries, How to Become - Simplilearn.com",Simplilearn.com,"Mon, 09 Mar 2026 07:00:00 GMT"
5,What are the high demand jobs in the UAE in the next 10 years? - Y-Axis Overseas Careers,Axis Overseas Careers,"Tue, 30 Sep 2025 07:00:00 GMT"
6,<a href='https://news.google.com/rss/articles/CBMi2AFBVV95cUxOYzN4NTdhXy1mcGxHaDl1NnVlcVdUdURzS0Jaa2s1WENKWlgyYzZ5bTJ4azA4Q2pscDZqRmNGT0t5ZVF1TFp0djdiNUNRd0pMS2lFT0lQWGFZLUIzV3BtZVNsbUFabm5oMkV5Z1JDNFM4Q0djNWxQTjJDNnJ4UklnY1ZjUEROdmlaYS1PSXplQTVqRmJjUEI3bW1meGphakRuTUxUWjJjVnVmTU9CWEllSm5hVnZYeUx2NUd0Umd5MHVPeGlOSkN6a0hyREdHSzgxVGpDY2xqRnPSAd4BQVVfeXFMUHBTQVNzSW9JSGJtYnE3WmRfZS12bS1hUXVVMEhSMjFvVDE1U3BpYmJnSUQtOGR3OEtnWVVkcFNBNVhIVHhMTUZYTWNQWXNiTUVnYWc1aHJtb2RpVld0c0stV0h6bVBTSU0zSzZWS1B3ek4yQkhsQnUwc05FcXJUSEFYU01qckNFczlYRnRXSkIzSnQyempzbXRzQ2xzOTIzWHpvRk1GU0o2LVFfS28zSWJpOUUtT0dDTGxnN0taUzBUX1NsM2JzMU1McFUtdmlNV2lVRXByV3NhaTgtMDVR?oc=5' target='_blank' rel='noopener'>The hottest jobs 5 years from now? The answers will surprise you - The Economic Times,The Economic Times,"Thu, 15 May 2025 07:00:00 GMT"
7,Top 20 Highest Paying Jobs in the World in 2026-27 - Shiksha.com,Shiksha.com,"Tue, 16 Dec 2025 12:56:35 GMT"
8,Top Career Options After BA - Simplilearn.com,Simplilearn.com,"Sat, 14 Feb 2026 08:00:00 GMT"
9,36 Entry-Level United Nations (UN) Jobs Open to All Nationalities (September 2025) - Global South Opportunities,Global South Opportunities,"Thu, 04 Sep 2025 07:00:00 GMT"
10,<a href='https://news.google.com/rss/articles/CBMingFBVV95cUxON2g3WnBoVEEzNUxGMTdQWlNGU3NMU2FtVkZfWUxncGJSUVZJOURIT3Y5djBSWFpxYXc5Y0wzckROSk9MZTN2VTRPNkExU1R1YVgzNjVNZG9laFBMY1ZFaVlzdGFtUE5wVEE5eW1WdUFxYnowRGtFQmRuX0FBOFZWaDhvQ2RnbV9QNmg0SzhNTU9UTU5NWmNvYlBsX2IzUdIBrgFBVV95cUxPdXVESDlYVnV2MS1FM3JwN3FVc2JkMHJ0Nmt3OXF0N2JyMldBbFA3V05tV3BJNkt0WnVYdHd4ajdLR0s5T2RwQmFHUW5yanhvVDg2aGZsbjVVMkNrZEZPdksxUUV4aUY3TXhCMklsank4ajZVQ2N1VkNqV1RIMl80dktCc3lMYnYyZHVXSVNVaWs4Z0ZNaGJVQjJSRDZVOUt4d3M3cGFEdlNvZDZVRFE?oc=5' target='_blank' rel='noopener'>Business Process Outsourcing (BPO) in the Philippines - Nexford University,Nexford University,"Mon, 26 Jan 2026 08:00:00 GMT"


Saved:
- out\google_jobs_basic_20260331-113709.csv
- out\google_jobs_basic_20260331-113709.json


In [5]:
# Deprecated cell from older one-cell version.
# Ignore this cell; use cells 1-4 above.
